# Day 1: Exploring Ontario's electricity demand

Before building any model, look at the data and find the patterns a model would need to learn. Run `python -m forecaster download` first.

Write down what you notice in the markdown cells as you go: these observations are what you'll talk about in interviews.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # so `forecaster` imports from the notebooks/ folder

import matplotlib.pyplot as plt
import pandas as pd
from forecaster.data import load_dataset

df = load_dataset()
df.describe()

## 1. The big picture: demand over the years

In [ ]:
daily = df['ontario_demand'].resample('D').mean()
ax = daily.plot(figsize=(12, 4), title='Average daily Ontario demand (MW)')
ax.set_ylabel('MW');

**What do you see?** (Seasonal peaks? Anything unusual in spring 2020?)

## 2. The daily cycle: summer vs winter

In [ ]:
season = df.index.month.map(lambda m: 'Summer (Jun-Aug)' if m in (6, 7, 8)
                          else ('Winter (Dec-Feb)' if m in (12, 1, 2) else None))
hourly = (df.assign(season=season, hour=df.index.hour)
            .dropna(subset=['season'])
            .groupby(['hour', 'season'])['ontario_demand'].mean().unstack())
ax = hourly.plot(figsize=(10, 4), title='Average demand by hour of day')
ax.set_ylabel('MW');

**What do you see?** (When is the peak in summer vs winter, and why might that be?)

## 3. The weekly cycle

In [ ]:
names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
weekly = df.groupby(df.index.dayofweek)['ontario_demand'].mean()
weekly.index = names
ax = weekly.plot(kind='bar', figsize=(8, 3.5), title='Average demand by day of week')
ax.set_ylabel('MW');

## 4. Demand vs temperature

Only works if the weather download succeeded.

In [ ]:
if 'temperature_c' in df.columns:
    sample = df.dropna().sample(min(5000, len(df.dropna())), random_state=0)
    ax = sample.plot.scatter(x='temperature_c', y='ontario_demand', alpha=0.2, figsize=(8, 5),
                             title='Hourly demand vs Toronto temperature')
    ax.set_ylabel('MW')
else:
    print('No temperature column: run the weather download.')

**What do you see?** (This shape is why the model uses separate heating and cooling degree features.)